<a href="https://colab.research.google.com/github/alviglio/AlbJupyters/blob/main/SivaGuenterPeterBook/Chapter02draft01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 2: A simple degree-day snow model

In this section, we build a very simple snow model to illustrate how snow accumulation and snowmelt can be represented in a discrete-time hydrological model.

The example is deliberately simple. We prescribe snowfall and temperature for 12 time steps:

- during the first 6 time steps, snowfall occurs and temperature is below freezing;
- during the last 6 time steps, snowfall stops and temperature is above freezing.

This creates an idealized sequence with two phases:

1. **Accumulation phase**: snow falls and is stored as snow water equivalent, or `SWE`.
2. **Melt phase**: temperature rises above the melt threshold and the stored snow begins to melt.

The goal is not only to obtain a working snow model, but also to show how small mistakes in the time-step water balance can produce physically impossible results.

In [ ]:
P_s <- c(100, 100, 100, 100, 100, 100,
         0, 0, 0, 0, 0, 0)
Tdeg <- c(-10, -10, -10, -10, -10, -10,
          10, 10, 10, 10, 10, 10)

The vectors `P_s` and `Tdeg` define the forcing data for the snow model.

`P_s` is snowfall in mm/timestep. In this example, 100 mm of snow falls during each of the first six time steps, and no snow falls afterward.

`Tdeg` is air temperature in degrees Celsius. The first six time steps are cold, at -10 °C, so snow should accumulate. The final six time steps are warm, at 10 °C, so snow should melt.

## First attempt: degree-day melt without constraints

The first model, `snowmodel01()`, applies a simple degree-day equation:

In [ ]:
snowmodel01 <- function (snowfall, temperature, k_DDF, Tmelt=0, SWE_0=0) {
  # snowfall = vector in mm/timestep
  # temperature = vector in degC
  # k_DDF = Degree Day Factor mm/(degC*timestep)
  tmax <- length(snowfall)
  snowmelt <- rep(NA, tmax)
  SWE <- rep(NA, tmax + 1)
   SWE[1] <- SWE_0
  for (t in 1:tmax) {
    snowmelt[t] <- k_DDF*(temperature[t] - Tmelt)
    SWE[t+1] <- SWE[t] + snowfall[t] - snowmelt[t]
  }
  output <- cbind(t=1:tmax, snowfall=round(snowfall, 1),
                  temperature=round(temperature, 1), SWE=round(SWE[-1], 1),
                  snowmelt=round(snowmelt, 1))
  return(output)
}
snowmodel01(P_s, Tdeg, k_DDF=30)

This is the basic snow-storage balance:

new SWE = old SWE + snowfall - snowmelt

However, this first version has a serious problem. When temperature is below Tmelt, the degree-day equation gives negative melt. For example, with T = -10 °C, Tmelt = 0 °C, and k_DDF = 15, the model calculates:

snowmelt = 15 * (-10 - 0) = -150 mm/timestep

Negative snowmelt is physically impossible. Instead of melting snow, the model is artificially adding water to the snowpack. This is why the first model is incorrect.

In [ ]:
plot_tabella <- function (tab, tab2=NULL, ts=1) {
  tab <- rbind(tab[nrow(tab),], tab)
  tab[1,'t'] <- 0
  add_month_axis <- function() {
    axis(1, at=seq(max(tab[,'t'])/12/2, max(tab[,'t']), by=max(tab[,'t'])/12),
         labels=month.abb, tick=F)
    axis(1, at=seq(0, max(tab[,'t']), by=max(tab[,'t'])/12),
         labels=F, tick=T)
    abline(h=0, lty=2)
  }
 layout(matrix(1:3, ncol=1))
  plot(tab[,'t'], tab[,'snowfall'], type='S', lwd=4, col='#ADD8E6',
       xlab='', ylab='snowfall (mm/timestep)', xaxt='n')
   add_month_axis()
  if (!is.null(tab2)) {
    tab2 <- rbind(tab2[nrow(tab2),], tab2); tab[2,'t'] <- 0
    lines(tab2[,'t'], tab2[,'snowfall'], type='S', lwd=4, col='#FF0000')
  }
  plot(tab[,'t'], tab[,'SWE']/ts, type='l', lwd=4, col='#ADD8E6',
       xlab='', ylab='SWE (mm)', xaxt='n')
   add_month_axis()
  if (!is.null(tab2)) {
    lines(tab2[,'t'], tab2[,'SWE']/ts, type='S', lwd=4, col='#FF0000')
  }
  plot(tab[,'t'], tab[,'snowmelt'], type='S', lwd=4, col='#ADD8E6',
       xlab='', ylab='snowmelt (mm/timestep)', xaxt='n')
   add_month_axis()
  if (!is.null(tab2)) {
    lines(tab2[,'t'], tab2[,'snowmelt'], type='S', lwd=4, col='#FF0000')
  }
}

options(repr.plot.width = 12, repr.plot.height = 12)
par(ps=22, mar=c(3.2, 4.5, 2.2, 1.2), mgp=c(2.5, 0.8, 0), tcl=-0.35, xaxs="i", yaxs="r", las=0)
plot_tabella(snowmodel01(rep(P_s, each=30), rep(Tdeg, each=30), k_DDF=30), ts=30)


## Second attempt: prevent negative snowmelt

The second model, `snowmodel02()`, fixes the first problem by preventing snowmelt from becoming negative:

In [ ]:
snowmodel02 <- function (snowfall, temperature, k_DDF, Tmelt=0, SWE_0=0) {
  # snowfall = vector in mm/timestep
  # temperature = vector in degC
  # k_DDF = Degree Day Factor mm/(degC*timestep)
  tmax <- length(snowfall)
  snowmelt <- rep(NA, tmax)
  SWE <- rep(NA, tmax + 1)
   SWE[1] <- SWE_0
  for (t in 1:tmax) {
    snowmelt[t] <- max(c(k_DDF*(temperature[t] - Tmelt), 0))
    SWE[t+1] <- SWE[t] + snowfall[t] - snowmelt[t]
  }
  output <- cbind(t=1:tmax, snowfall=round(snowfall, 1),
                  temperature=round(temperature, 1), SWE=round(SWE[-1], 1),
                  snowmelt=round(snowmelt, 1))
  return(output)
}
snowmodel02(P_s, Tdeg, k_DDF=30)

Now, if the temperature is below the melt threshold, the calculated melt is replaced by zero.

This gives a more realistic melt equation:

snowmelt = max(degree-day melt, 0)

The snowpack is still updated with:

SWE[t + 1] <- SWE[t] + snowfall[t] - snowmelt[t]

This fixes the negative-melt problem, but introduces another issue. During warm periods, the model can melt more snow than is actually available in the snowpack. As a result, SWE can become negative.

Negative snow water equivalent is also physically impossible. The snowpack cannot contain less than zero water.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 12)
par(ps=22, mar=c(3.2, 4.5, 2.2, 1.2), mgp=c(2.5, 0.8, 0), tcl=-0.35, xaxs="i", yaxs="r", las=0)
plot_tabella(snowmodel02(rep(P_s, each=30), rep(Tdeg, each=30), k_DDF=30), ts=30)

## Third attempt: prevent negative SWE

The third model, `snowmodel03()`, adds another constraint. After updating the snowpack, it forces `SWE` to remain non-negative:

In [ ]:
snowmodel03 <- function (snowfall, temperature, k_DDF, Tmelt=0, SWE_0=0) {
  # snowfall = vector in mm/timestep
  # temperature = vector in degC
  # k_DDF = Degree Day Factor mm/(degC*timestep)
  tmax <- length(snowfall)
  snowmelt <- rep(NA, tmax)
  SWE <- rep(NA, tmax + 1)
   SWE[1] <- SWE_0
  for (t in 1:tmax) {
    snowmelt[t] <- max(c(k_DDF*(temperature[t] - Tmelt), 0))
    SWE[t+1] <- max(c(SWE[t] + snowfall[t] - snowmelt[t], 0))
  }
  output <- cbind(t=1:tmax, snowfall=round(snowfall, 1),
                  temperature=round(temperature, 1), SWE=round(SWE[-1], 1),
                  snowmelt=round(snowmelt, 1))
  return(output)
}
snowmodel03(P_s, Tdeg, k_DDF=30)

This ensures that the snow storage never drops below zero.

However, the model still has a subtle inconsistency. The reported snowmelt can be larger than the amount of snow that was available to melt during that time step. The model hides the problem by setting the final SWE to zero, but the melt flux itself is still physically unrealistic.

For example, if only 50 mm of snow is available, the model should not report 150 mm of snowmelt. The snowpack can be emptied, but it cannot produce more meltwater than the snow it contains.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 12)
par(ps=22, mar=c(3.2, 4.5, 2.2, 1.2), mgp=c(2.5, 0.8, 0), tcl=-0.35, xaxs="i", yaxs="r", las=0)
plot_tabella(snowmodel03(rep(P_s, each=30), rep(Tdeg, each=30), k_DDF=30), ts=30)

## Fourth attempt: limit snowmelt by available snow

The fourth model, `snowmodel04()`, fixes this problem by limiting snowmelt to the amount of snow available at the beginning of the time step:

In [ ]:
snowmodel04 <- function (snowfall, temperature, k_DDF, Tmelt=0, SWE_0=0) {
  # snowfall = vector in mm/timestep
  # temperature = vector in degC
  # k_DDF = Degree Day Factor mm/(degC*timestep)
  tmax <- length(snowfall)
  snowmelt <- rep(NA, tmax)
  SWE <- rep(NA, tmax + 1)
   SWE[1] <- SWE_0
  for (t in 1:tmax) {
    snowmelt[t] <- min(max(c(k_DDF*(temperature[t] - Tmelt), 0)), SWE[t])
    SWE[t+1] <- max(c(SWE[t] + snowfall[t] - snowmelt[t], 0))
  }
  output <- cbind(t=1:tmax, snowfall=round(snowfall, 1),
                  temperature=round(temperature, 1), SWE=round(SWE[-1], 1),
                  snowmelt=round(snowmelt, 1))
  return(output)
}
snowmodel04(P_s, Tdeg, k_DDF=30)

This line combines two physical constraints:

Snowmelt cannot be negative:
snowmelt >= 0
Snowmelt cannot exceed the snow water equivalent available at the start of the time step:
snowmelt <= SWE[t]

The snowpack is then updated with:

SWE[t + 1] <- max(c(SWE[t] + snowfall[t] - snowmelt[t], 0))

This version is physically more consistent. The model now allows snow to accumulate during cold periods and melt during warm periods, while preventing both negative snowmelt and negative snow storage.

A point that may look confusing at first is that snowmelt can appear larger than the final SWE shown in the output table. This is not necessarily wrong, because snowmelt occurs during the time step, while the reported SWE is the storage remaining at the end of the time step.

For example, if the snowpack starts the time step with 150 mm of SWE and melts 150 mm, the end-of-step SWE is zero. The melt is larger than the final SWE, but it is not larger than the initial SWE available during the time step.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 12)
par(ps=22, mar=c(3.2, 4.5, 2.2, 1.2), mgp=c(2.5, 0.8, 0), tcl=-0.35, xaxs="i", yaxs="r", las=0)
plot_tabella(snowmodel04(rep(P_s, each=30), rep(Tdeg, each=30), k_DDF=30), ts=30)

The final part of the code plots the model results in three panels:

1. snowfall input;
2. snow water equivalent, `SWE`;
3. snowmelt output.

The first panel shows the prescribed snowfall. Snowfall occurs during the first six time steps and then stops.

The second panel shows the evolution of snow water equivalent. During the cold period, snowfall accumulates and SWE increases. During the warm period, the snowpack melts and SWE decreases until it reaches zero.

The third panel shows snowmelt. There is no melt during the cold period because temperature is below the melt threshold. Once temperature rises above 0 °C, melt begins. Melt continues only while snow is available.

Together, the three panels show how the model converts snowfall and temperature into snow accumulation and snowmelt.